# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [2]:
import pandas as pd
# read the sentence data
df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx")
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,I was rushed into the [PRESSION] two weeks ago...,1
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",I was acutely under a painful pressure on the ...,2
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,It started 1 hour after dinner.,3
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...","I had migraines all day, so I hadn't eaten much.",4
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...","I got very nauseous, vomiting, dizzy and rejuv...",5


In [3]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [4]:
# print the size of the data
len(df)
# There are 41268 unique sentences 

41268

### 1.2. Import the list of stopwords

In [5]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

/workspace/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-09 11:23:02.122126: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Batches:   0%|          | 0/1290 [00:00<?, ?it/s]

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

/workspace/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-09 08:07:22.068906: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### 1.5 functions to pre-configure burtopic model

In [35]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')

In [24]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer


def tune_umap(n_neighbors, n_components, min_dist):
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist, metric='cosine', random_state=42)
    return umap_model

def tune_hdb(min_cluster_size, min_samples, cluster_selection_epsilon):
    hdb_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    return hdb_model

def build_bertopic(embedding_model, umap_model, cluster_model, vectorizer_model, embeddings, docs):
    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'),
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(docs, embeddings)
    

    # Return only essential results
    return topic_model

## 2. Build BERTopic Model

In [9]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [25]:
# build the model with the preconfigured settings in 1.5
# use a empirical set of parameters 
umap_model = tune_umap(15, 5, 0.05)
hdb_model = tune_hdb(20, 10, 0.0)
topic_model = build_bertopic(embedding_model=embedding_model, umap_model=umap_model, cluster_model=hdb_model, vectorizer_model=vectorizer_model, embeddings=embeddings, docs=sentences)

2025-02-09 11:40:26,171 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-09 11:41:49,026 - BERTopic - Dimensionality - Completed ✓
2025-02-09 11:41:49,031 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 11:41:53,946 - BERTopic - Cluster - Completed ✓
2025-02-09 11:41:53,964 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 11:41:54,822 - BERTopic - Representation - Completed ✓


In [26]:
# examine topic information
pd.set_option("max_colwidth", 200) # adjust column width 
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,15193,-1_recept_afspraak_vraag_medicatie,"[recept, afspraak, vraag, medicatie, gaat, krijg, vragen, klachten, mogelijk, volgende]","[Kunnen jullie een nieuw recept maken voor mijn medicatie Thiosix voor de komende maanden?, Ik wilde wel nog vragen of ik dan alstublieft een nieuw recept mag voor de medicijnen., ja had graag rec..."
1,0,3414,0_bloed_prikken_bloedprikken_ontlasting,"[bloed, prikken, bloedprikken, ontlasting, slijm, bloeduitslagen, laten, geprikt, ingeleverd, inleveren]","[Op 21 april heb ik bloed laten prikken., Moet ik misschien bloed laten prikken?, Ik heb maandag bloed laten prikken.]"
2,1,1569,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, gynaecoloog, medische, reumatoloog]","[Zou ik naar de huisarts moeten?, Bij de huisarts of in het ziekenhuis?, Ik ben afgelopen vrijdag weer bij de huisarts geweest.]"
3,2,723,2_test_testen_proberen_quanton,"[test, testen, proberen, quanton, zelftest, cal, gedaan, geprobeerd, thuis, calprotectine]","[Moet ik deze test nu doen ?, De test is onder de, Zal de test doen.]"
4,3,623,3_medicatie_medicijnen_medicijn_apotheek,"[medicatie, medicijnen, medicijn, apotheek, recept, medicijngebruik, ontvangen, voorbereiding, gebruik, opgehaald]","[Ik heb een vraag betreffende medicatie., Graag zou ik andere medicatie willen., Ik vroeg me af of dit met de medicatie te maken heeft?]"
...,...,...,...,...,...
264,263,20,263_telefonisch_telefonische_gemist_krjgen,"[telefonisch, telefonische, gemist, krjgen, benaderd, intakegesprek, consult, mriscan, veronderstelling, vrijdagavond]","[Ik had eigenlijk afgelopen maandag een telefonisch consult., / N.a.v.telefonisch contact afgelopen vrijdag ben ik naar het aan geweest., medewerker van MijnIBDcoach, Afgelopen donderdag 19 maart..."
265,264,20,264_vraagjes_vragen_mexico_hierover,"[vraagjes, vragen, mexico, hierover, hiervan, vertrek, aanleiding, dingen, besproken, hiervoor]","[Ik heb even twee vraagjes., Ik heb 2 vraagjes., Ik heb 2 vraagjes.]"
266,265,20,265_gaten_houden_container_nauwlettend,"[gaten, houden, container, nauwlettend, meebrengen, inhouden, neerleggen, computer, vul, aanzien]","[Ik zal het in de gaten houden., Ja ik zal hem in de gaten houden, Ik zal het in de gaten houden.]"
267,266,20,266_alvast_bedankt_plaatsvindt_verneem,"[alvast, bedankt, plaatsvindt, verneem, tel, fax, enorm, hopelijk, apotheek, graag]","[Bedankt alvast, (), Alvast bedankt !, Alvast bedankt!]"


In [27]:
# Calculate the coherence score of the model
# Note: to evaluate with coherence score, the n_gram of the vectorizer_model has to be (1, 1)
def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

In [28]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
top_words = get_top_words(topic_model)
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score}")


Coherence Score: 0.3504502636149212


In [29]:
# reduce outliers of this topic model and see what changes to the coherence score
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 16/16 [00:02<00:00,  6.38it/s]


In [30]:
topic_model.update_topics(sentences, topics=new_topics)

2025-02-09 11:42:56,137 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [34]:
topic_model.update_topics(vectorizer_model=vectorizer_model, docs=sentences)

In [37]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
top_words = get_top_words(topic_model)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score of Model after Outlier Reduction: {coherence_score}")


Coherence Score of Model after Outlier Reduction: 0.35114895874259366


In [36]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,240,-1_azethoprine_ouder_handigst_vriendleijke,"[azethoprine, ouder, handigst, vriendleijke, voorgekomen, verhelpen, vakjes, tijdelijks, permitteren, wachttijden]","[Kunnen jullie een nieuw recept maken voor mijn medicatie Thiosix voor de komende maanden?, Ik wilde wel nog vragen of ik dan alstublieft een nieuw recept mag voor de medicijnen., ja had graag rec..."
1,0,3555,0_bloed_prikken_bloedprikken_slijm,"[bloed, prikken, bloedprikken, slijm, ontlasting, laten, bloeduitslagen, geprikt, ingeleverd, inleveren]","[Op 21 april heb ik bloed laten prikken., Moet ik misschien bloed laten prikken?, Ik heb maandag bloed laten prikken.]"
2,1,1682,1_huisarts_arts_dokter_mdl,"[huisarts, arts, dokter, mdl, ziekenhuis, contact, verpleegkundige, reumatoloog, gynaecoloog, medische]","[Zou ik naar de huisarts moeten?, Bij de huisarts of in het ziekenhuis?, Ik ben afgelopen vrijdag weer bij de huisarts geweest.]"
3,2,788,2_test_testen_proberen_quanton,"[test, testen, proberen, quanton, cal, gedaan, zelftest, geprobeerd, thuis, calprotectine]","[Moet ik deze test nu doen ?, De test is onder de, Zal de test doen.]"
4,3,756,3_medicijnen_medicatie_medicijn_apotheek,"[medicijnen, medicatie, medicijn, apotheek, voorbereiding, recept, bestellen, opgehaald, gebruik, medicijngebruik]","[Ik heb een vraag betreffende medicatie., Graag zou ik andere medicatie willen., Ik vroeg me af of dit met de medicatie te maken heeft?]"
...,...,...,...,...,...
264,263,78,263_telefonische_omgezet_veronderstelling_vrijdagavond,"[telefonische, omgezet, veronderstelling, vrijdagavond, probeer, telefonisch, consult, stond, afspraak, mriscan]","[Ik had eigenlijk afgelopen maandag een telefonisch consult., / N.a.v.telefonisch contact afgelopen vrijdag ben ik naar het aan geweest., medewerker van MijnIBDcoach, Afgelopen donderdag 19 maart..."
265,264,84,264_vragen_vraagjes_hierover_aanleiding,"[vragen, vraagjes, hierover, aanleiding, invullen, mening, endeldarm, scopie, opmerkingen, vervolgonderzoek]","[Ik heb even twee vraagjes., Ik heb 2 vraagjes., Ik heb 2 vraagjes.]"
266,265,69,265_gaten_houden_vul_neerleggen,"[gaten, houden, vul, neerleggen, gang, gehouden, hou, verhogen, rekening, gele]","[Ik zal het in de gaten houden., Ja ik zal hem in de gaten houden, Ik zal het in de gaten houden.]"
267,266,77,266_alvast_vriendelijk_bedankt_klaarmaken,"[alvast, vriendelijk, bedankt, klaarmaken, mezalazine, chirurg, computer, plaatsvindt, voorleggen, overlegd]","[Bedankt alvast, (), Alvast bedankt !, Alvast bedankt!]"


## 3. Fine-Tune the Model

In [9]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)


def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [10]:
import itertools

# Define parameter ranges
range_n_neighbors = [5, 10, 20, 35, 50, 75, 100]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.01, 0.05, 0.1]

# Generate all possible combinations of the parameters
umap_combinations = list(itertools.product(range_n_neighbors, range_n_components, range_min_dist))

In [16]:
import random
from joblib import Parallel, delayed

def tune_bertopic_umap_random(embedding_model, vectorizer_model, embeddings, sentences, n_iter, tried_umap_combinations):
    """
    Randomized search over UMAP and HDBSCAN hyperparameters to find the best model based on coherence score.
    
    Args:
        embedding_model: The embedding model to use for topic modeling.
        vectorizer_model: The vectorizer model to use for tokenization.
        embeddings: The embeddings of the sentences.
        sentences: The sentences (documents) for topic modeling.
        n_iter: The number of random iterations to perform in the randomized search.
        
    Returns:
        best_topic_model: The best topic model based on coherence score.
        best_coherence_score: The best coherence score obtained during the search.
    """
    
    # Define ranges for the UMAP parameters
    range_n_neighbors = [5, 10, 20, 35, 50, 75, 100]
    range_n_components = [2, 3, 4, 5, 6]
    range_min_dist = [0.0, 0.01, 0.05, 0.1]
    
    best_coherence_score = -float('inf')
    best_topic_model = None

    # Randomized search loop
    for _ in range(n_iter):
        # Sample random UMAP parameters
        n_neighbors = random.choice(range_n_neighbors)
        n_components = random.choice(range_n_components)
        min_dist = random.choice(range_min_dist)

        # Tune UMAP and HDBSCAN
        umap_model = tune_umap(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist)
        hdb_model = tune_hdb(min_cluster_size=20, min_samples=10, cluster_selection_epsilon=0.0)

        # record the combinations that are already tried
        tried_umap_combinations.append((n_neighbors, n_components, min_dist))
        
        # Build topic model
        current_topic_model = build_bertopic(
            embedding_model=embedding_model, 
            umap_model=umap_model, 
            cluster_model=hdb_model, 
            vectorizer_model=vectorizer_model, 
            embeddings=embeddings, 
            docs=sentences
        )
        
        # Calculate coherence score for the current model
        current_coherence_score = calculate_c_v(current_topic_model)
        print(f"Coherence score: {current_coherence_score} for UMAP params: {n_neighbors}, {n_components}, {min_dist}")
        
        # Check if the current model has a better coherence score
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = current_topic_model
            best_combination = (n_neighbors, n_components, min_dist)
            
    return best_topic_model, best_coherence_score, best_combination, tried_umap_combinations

# Example usage
best_model, best_score, best_combination, tried_umap_combinations = tune_bertopic_umap_random(embedding_model=embedding_model, vectorizer_model=vectorizer_model, embeddings=embeddings, sentences=sentences, n_iter=20, tried_umap_combinations=[])
print("Best UMAP Parameters:", best_combination)
print("Best Coherence Score:", best_score)


2025-02-09 08:14:20,580 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-09 08:15:46,515 - BERTopic - Dimensionality - Completed ✓
2025-02-09 08:15:46,520 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 08:15:51,984 - BERTopic - Cluster - Completed ✓
2025-02-09 08:15:52,004 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 08:15:52,860 - BERTopic - Representation - Completed ✓
2025-02-09 08:16:36,522 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3447880155006443 for UMAP params: 20, 5, 0.05


2025-02-09 08:17:39,967 - BERTopic - Dimensionality - Completed ✓
2025-02-09 08:17:39,971 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 08:17:42,897 - BERTopic - Cluster - Completed ✓
2025-02-09 08:17:42,915 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 08:17:43,888 - BERTopic - Representation - Completed ✓
2025-02-09 08:18:36,944 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3408803691743068 for UMAP params: 10, 5, 0.0


2025-02-09 08:22:25,738 - BERTopic - Dimensionality - Completed ✓
2025-02-09 08:22:25,743 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 08:22:29,201 - BERTopic - Cluster - Completed ✓
2025-02-09 08:22:29,219 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 08:22:30,022 - BERTopic - Representation - Completed ✓
2025-02-09 08:23:04,384 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3515256767991325 for UMAP params: 100, 5, 0.01


2025-02-09 08:23:36,732 - BERTopic - Dimensionality - Completed ✓
2025-02-09 08:23:36,736 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 08:23:38,520 - BERTopic - Cluster - Completed ✓
2025-02-09 08:23:38,541 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 08:23:39,588 - BERTopic - Representation - Completed ✓
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fa75ac5f550>>
Traceback (most recent call last):
  File "/workspace/mijnidbcoachnlp/venv/lib/python3.8/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
# save best model 
